# Soluciones — Clase 28: Ética, sesgo y privacidad en datos

Este notebook contiene las soluciones completas a los ejercicios de la clase.

> **Nota para el docente:** Comparte este notebook solo después de que los estudiantes hayan intentado resolver los ejercicios por su cuenta.

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.ensemble import RandomForestClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score
from sklearn.preprocessing import LabelEncoder

plt.style.use('seaborn-v0_8')

# Recrear el dataset
np.random.seed(42)
n = 300
cursos = np.random.choice(['A', 'B', 'C'], size=n, p=[0.4, 0.4, 0.2])
edades = np.random.randint(18, 35, size=n)
nota_mate = np.where(
    cursos == 'A', np.random.normal(7.5, 1.2, n),
    np.where(cursos == 'B', np.random.normal(7.0, 1.3, n), np.random.normal(5.8, 1.5, n))
).clip(1, 10)
nota_lengua = np.where(
    cursos == 'A', np.random.normal(7.2, 1.1, n),
    np.where(cursos == 'B', np.random.normal(6.8, 1.2, n), np.random.normal(5.5, 1.4, n))
).clip(1, 10)
asistencia = np.where(cursos == 'C', np.random.uniform(0.55, 0.85, n), np.random.uniform(0.70, 0.99, n))
prob_aprobar = (nota_mate * 0.4 + nota_lengua * 0.4 + asistencia * 10 * 0.2) / 10
aprobado = (np.random.uniform(0, 1, n) < prob_aprobar).astype(int)

df = pd.DataFrame({
    'curso': cursos, 'edad': edades,
    'nota_matematicas': nota_mate.round(1),
    'nota_lengua': nota_lengua.round(1),
    'asistencia': asistencia.round(2),
    'aprobado': aprobado
})
print('Dataset creado:', df.shape)

## Solución Ejercicio 2: Analizar distribución del dataset

In [ ]:
# a) Distribución por curso
print('=== a) Estudiantes por curso ===')
print(df['curso'].value_counts())

# b) Tasa de aprobación por curso
print('\n=== b) Tasa de aprobación por curso ===')
tasas_curso = df.groupby('curso')['aprobado'].mean().round(3)
print(tasas_curso)

# c) Tasa de aprobación por rango de edad
print('\n=== c) Tasa de aprobación por rango de edad ===')
df['rango_edad'] = pd.cut(
    df['edad'],
    bins=[0, 20, 25, 30, 100],
    labels=['<20', '20-25', '25-30', '30+']
)
print(df.groupby('rango_edad', observed=True)['aprobado'].mean().round(3))

# Visualización
fig, axes = plt.subplots(1, 3, figsize=(15, 4))

df['curso'].value_counts().plot(kind='bar', ax=axes[0], color='#4C72B0')
axes[0].set_title('Estudiantes por curso')
axes[0].set_xlabel('Curso')

tasas_curso.plot(kind='bar', ax=axes[1], color='#55A868')
axes[1].set_title('Tasa de aprobación por curso')
axes[1].set_ylim(0, 1)
axes[1].set_xlabel('Curso')

df.groupby('rango_edad', observed=True)['aprobado'].mean().plot(
    kind='bar', ax=axes[2], color='#C44E52'
)
axes[2].set_title('Tasa aprobación por edad')
axes[2].set_ylim(0, 1)

plt.tight_layout()
plt.show()

## Solución Ejercicio 3: Calcular impacto dispar

In [ ]:
tasas = df.groupby('curso')['aprobado'].mean()
tasa_max = tasas.max()
tasa_min = tasas.min()
razon = tasa_min / tasa_max

print(f'Tasa máxima (curso {tasas.idxmax()}): {tasa_max:.1%}')
print(f'Tasa mínima (curso {tasas.idxmin()}): {tasa_min:.1%}')
print(f'Razón de impacto dispar: {razon:.2f}')

if razon < 0.8:
    print('\n⚠️  Hay impacto dispar significativo.')
    print('Interpretación: El curso C tiene una tasa de aprobación que es')
    print(f'solo el {razon:.0%} de la tasa del curso más exitoso.')
    print()
    print('¿Esto significa que el modelo es injusto?')
    print('No necesariamente. Las diferencias pueden deberse a:')
    print('- Diferencias reales en preparación académica')
    print('- Diferencias en recursos disponibles (sesgo estructural)')
    print('- Sesgo en la recolección de datos')
    print('Hay que investigar las causas antes de concluir.')

## Solución Ejercicio 4: Evaluar modelo por subgrupos

In [ ]:
le = LabelEncoder()
df['curso_encoded'] = le.fit_transform(df['curso'])

features = ['nota_matematicas', 'nota_lengua', 'asistencia', 'edad']
X = df[features]
y = df['aprobado']

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.25, random_state=42
)
grupos_test = df.loc[X_test.index, 'curso']

modelo = RandomForestClassifier(n_estimators=100, random_state=42)
modelo.fit(X_train, y_train)
predicciones = modelo.predict(X_test)

print(f'Precisión GLOBAL: {accuracy_score(y_test, predicciones):.2%}')
print()
print('=== Precisión por curso ===')

for curso in sorted(df['curso'].unique()):
    mask = grupos_test == curso
    if mask.sum() > 0:
        acc = accuracy_score(y_test[mask], predicciones[mask])
        n_grupo = mask.sum()
        print(f'Curso {curso}: {acc:.2%} (n={n_grupo})')

print()
print('OBSERVACIÓN: Si el modelo tiene menor precisión para el curso C,')
print('puede ser porque tiene menos ejemplos de entrenamiento (sesgo de muestreo)')
print('o porque el patrón es más difícil de capturar para ese grupo.')

## Solución Ejercicio 6: Anonimización de datos

In [ ]:
import hashlib

datos = pd.DataFrame({
    'nombre': ['Ana García', 'Juan Pérez', 'María López', 'Carlos Ruiz', 'Sofía Mora'],
    'dni': ['12345678A', '87654321B', '11223344C', '99887766D', '55443322E'],
    'edad': [25, 32, 28, 45, 22],
    'departamento': ['Ventas', 'IT', 'Marketing', 'IT', 'Ventas'],
    'salario': [35000, 45000, 38000, 62000, 30000]
})

print('ANTES de anonimizar:')
print(datos)

# Técnica 1: Eliminar identificadores directos
datos_sin_id = datos.drop(columns=['nombre', 'dni'])
print('\nDespués de eliminar nombre y DNI:')
print(datos_sin_id)

# Técnica 2: Generalizar el salario
datos_sin_id['rango_salario'] = pd.cut(
    datos_sin_id['salario'],
    bins=[0, 35000, 45000, 65000, float('inf')],
    labels=['<35k', '35k-45k', '45k-65k', '>65k']
)
datos_anonimos = datos_sin_id.drop(columns=['salario'])
print('\nCon salario generalizado en rangos:')
print(datos_anonimos)

print()
print('NOTA: En un dataset pequeño como este, la combinación de')
print('departamento + edad podría ser suficiente para identificar a alguien.')
print('La anonimización perfecta en datasets pequeños es muy difícil.')

## Reflexión final

Los ejercicios de esta clase muestran que:

1. **Los datos raramente son neutros** — siempre reflejan decisiones de quién los recopiló, cuándo y con qué propósito.

2. **La precisión global puede ocultar inequidades** — siempre evalúa el rendimiento por subgrupos.

3. **Anonimizar es más difícil de lo que parece** — eliminar el nombre no es suficiente en la mayoría de los casos.

4. **La transparencia tiene valor práctico** — un modelo que puede explicarse es más fácil de auditar, corregir y mejorar.

5. **La ética es iterativa** — no es un checklist que se hace una vez; es una práctica continua a lo largo de todo el proyecto.